In [ ]:
import sys
sys.path.append("..")
import utils_for_cluster_first_try as utils
import utils_for_rings_and_discs as gen_and_plot_utils

import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

import importlib
importlib.reload(utils)


# Analyze the results after training

### Load the dataset

In [ ]:
import pickle
import io
importlib.reload(utils)

from pathlib import Path

DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_dataset.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict = pickle.load(f)

### Load the results of the training

In [ ]:
DICT_LOAD_PATH = Path.cwd() / "dict_running_VAE.pkl"

class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
        else:
            return super().find_class(module, name)

with open(DICT_LOAD_PATH, "rb") as f:
    output_dict = CPU_Unpickler(f).load()

### Plot the results of training

In [ ]:
train_dict = output_dict["train_dict"]
utils.graph_training_output(train_dict,model_name="unsup_VAE",num_skip_first_epoch=0000)

zhats = train_dict["zhats"]


encoder_mean = output_dict["encoder_mean"]
decoder_mean = output_dict["decoder_mean"]
Yhat = output_dict["Yhat"]
Zhat = output_dict["Zhat"]
model = output_dict["model"]
optimizer = output_dict["optimizer"]
B_est = output_dict["B_est"]
#Sigma_Z_est = output_dict["Sigma_Z_est"]
#Sigma_X_est = output_dict["Sigma_X_est"]
H_est = output_dict["H_est"]
T_est = output_dict["T_est"]
lambdas_est = output_dict["lambdas_est"]

X = dataset_dict["X"]
Y = dataset_dict["Y"]
Z_standard = dataset_dict["Z_standard"]
Z = dataset_dict["Z"]

print(Zhat.shape)
print(H_est.shape)

U = Zhat@H_est
V = X@T_est

In [ ]:
#print(B_est)
print(T_est)
print(H_est)



### Plot original versus reconstructed meshes

In [ ]:
importlib.reload(gen_and_plot_utils)

Y_recon_no_noise = decoder_mean(torch.tensor(Zhat,dtype=torch.float32)).cpu().detach().numpy()
gen_and_plot_utils.plot_reconstruction(Y, Y_recon_no_noise,num_samples = 3)

In [ ]:
importlib.reload(utils)

# we want to plot the latent space
colors_1 = Z[:,2]
colors_2 = Z[:,3]
#utils.plot_latent_variables(U, colors_1, colors_2, color_1_label="radius_of_hole", color_2_label="width_of_disc", H_est=None)

utils.plot_latent_variables_d_greater_than_2(Zhat, Z)

### How well does the VAE generate new images?

In [ ]:
num_samples = 15  # Number of samples to generate

#Using a standard gaussian performs poorly as you might expect
#gen_Z = np.random.normal(0, 1, (num_samples, 2))  # 2D standard Gaussian

#instead, we will sample from a KDE learned on Zhat
from sklearn.neighbors import KernelDensity
kde = KernelDensity(kernel='gaussian', bandwidth=0.1).fit(Zhat)
gen_Z = kde.sample(num_samples)  # Sample from the KDE

# Use the Y_decoder_mean to get the Y reconstructions
gen_Z_tensor = torch.tensor(gen_Z, dtype=torch.float32)
gen_Y_recon = decoder_mean(gen_Z_tensor).cpu().detach().numpy()

# Reshape the generated Y reconstructions to match the image size
gen_Y_recon = gen_Y_recon.reshape(num_samples, int(np.sqrt(gen_Y_recon.shape[1])), int(np.sqrt(gen_Y_recon.shape[1])))
plt.figure(figsize=(10, 10))
for i in range(num_samples):
    plt.subplot(5, 3, i + 1)
    plt.imshow(gen_Y_recon[i], cmap='gray', origin='lower')
    plt.axis('off')
    plt.title(f'Generated Image {i}')
plt.tight_layout()
plt.show()


# let's try to visualize the latent space Zhat
plt.figure(figsize=(8, 6))
scatter = plt.scatter(Zhat[:, 0], Zhat[:, 1], c=Z[:,0], cmap='viridis', alpha=0.7)
plt.colorbar(label='r_1 (Radius of the hole) (0 <--> disc)')
plt.xlabel('Zhat[0]')
plt.ylabel('Zhat[1]')
plt.title('Latent Space Zhat Visualization')
plt.grid(True)

# also plot the generated points
plt.scatter(gen_Z[:, 0], gen_Z[:, 1], c='red', marker='x', label='Generated Points', alpha=0.5)
plt.legend()
# Annotate the generated points with their indices
for i, (x, y, _, _) in enumerate(gen_Z):
    plt.text(x, y, str(i), fontsize=10, ha='right', va='bottom', color='red')


### In-sample correlation attained

In [ ]:
utils.plot_canonical_variables(U - np.mean(U, axis=0), V - np.mean(V, axis=0), Z_standard[:, 2], Z_standard[:, 3])

# Out of sample results

In [ ]:
importlib.reload(utils)

#large validation set
DATASET_LOAD_PATH = Path.cwd() / "rings_and_discs_validation_dataset.pkl"

with open(DATASET_LOAD_PATH, "rb") as f:
    dataset_dict_val = pickle.load(f)


Y_val = dataset_dict_val["Y_val"]
X_val = dataset_dict_val["X_val"]
Z_standard_val = dataset_dict_val["Z_standard_val"]
Z_val = dataset_dict_val["Z_val"]
colors_1_val = Z_standard_val[:,2]
colors_2_val = Z_standard_val[:,3]
val_dataset = utils.XYDataset(X_val,Y_val)

out_of_sample_correlations = utils.out_of_sample_correlation(val_dataset,encoder_mean=encoder_mean,H=H_est,T=T_est)
print("Out-of-sample canonical correlations:", out_of_sample_correlations)
print("sum:", np.sum(np.abs(out_of_sample_correlations)))

Zhat_val = encoder_mean(val_dataset.Y).detach().numpy()
U_val = Zhat_val@H_est
V_val = X_val@T_est

utils.plot_canonical_variables(U_val - np.mean(U_val, axis=0), V_val - np.mean(V_val, axis=0), colors_1_val, colors_2_val)

In [ ]:
utils.out_of_sample_reconstruction_error(val_dataset,encoder_mean,decoder_mean)

In [ ]:
# Used to save the VAE model for use with the normalizing flow model

X = dataset_dict["X"]
Z = dataset_dict["Z"]
Z_standard= dataset_dict["Z_standard"]
Y = dataset_dict["Y"]
Zhat = output_dict["Zhat"]
Y_val = dataset_dict["Y_val"]
X_val = dataset_dict["X_val"]
Z_val = dataset_dict["Z_val"]
Z_standard_val = dataset_dict["Z_standard_val"]
dataset_dict = {
    "X": X,
    "Y": Y,
    "Z": Z,
    "Z_standard": Z_standard,
    "Zhat": Zhat,
    "Y_val": Y_val,
    "X_val": X_val,
    "Z_val": Z_val,
    "Z_standard_val": Z_standard_val
}

from pathlib import Path

DATASET_SAVE_PATH = Path.cwd() / "rings_and_discs_dataset_for_NF.pkl"
import pickle
with open(DATASET_SAVE_PATH, "wb") as f:
    pickle.dump(dataset_dict, f)
